In [1]:
# Cell 1: 环境
import os, sys, json
current_dir = os.getcwd()
project_root = os.path.abspath(os.path.join(current_dir, '..'))
if os.path.exists(os.path.join(project_root, 'config.py')):
    sys.path.insert(0, project_root)

from config import get_driver
from src.ci.competitor_brief import generate_competitor_brief
from src.engineering_intelligence.technology_brief import generate_technology_brief
from src.engineering_intelligence.strategy_classifier import get_all_strategies

driver = get_driver()
print("✅ 环境就绪")

✅ 环境就绪


In [2]:
# Cell 2: Competitor Brief + 三个质量能力区块
with driver.session() as s:
    org = s.run("""
        MATCH (o:Organization)-[:DEVELOPS]->(:DevelopmentProgram)
        RETURN o.organization_id AS oid, o.canonical_name AS name
        ORDER BY o.canonical_name LIMIT 1
    """).single()

brief = generate_competitor_brief(driver, org["oid"], days_back=730, persist=True)
bid = brief["brief_id"]
print(f"✅ Competitor Brief: {bid}\n")

with driver.session() as s:
    row = s.run("""
        MATCH (b:IntelligenceProduct {brief_id: $bid})
        RETURN b.source_credibility_breakdown AS sc,
               b.entity_resolution_hints AS er,
               b.conflict_summary AS cf
    """, bid=bid).single()

print("=== source_credibility_breakdown ===")
print(json.dumps(json.loads(row["sc"] or "{}"), ensure_ascii=False, indent=2))

print("\n=== entity_resolution_hints ===")
er = json.loads(row["er"] or "{}")
print(f"  候选数: {er.get('candidate_count', 0)}")
for c in er.get("duplicate_candidates", [])[:3]:
    print(f"    {c.get('candidate_canonical_name')} score={c.get('score')}")

print("\n=== conflict_summary ===")
cf = json.loads(row["cf"] or "{}")
print(f"  total_conflicts: {cf.get('total_conflicts', 0)}")

print("\n=== recommended_next_steps ===")
for s_ in brief.get("recommended_next_steps", []):
    print(f"  - {s_}")

✅ Competitor Brief: CI:BRIEF:DB3A21A0

=== source_credibility_breakdown ===
{
  "total_sources": 1,
  "by_tier": {
    "primary": 0,
    "secondary": 1,
    "tertiary": 0,
    "unknown": 0
  },
  "avg_score": 67.0,
  "scored_sources": 1,
  "note": "来源可信度评分只反映来源渠道本身的可靠性，不代表内容正确性。"
}

=== entity_resolution_hints ===
  候选数: 0

=== conflict_summary ===
  total_conflicts: 0

=== recommended_next_steps ===
  - 建议跟进近期重大事件，评估对内部战略的影响
  - 对手拥有 3 个专利族，建议启动 FTO 初筛
  - 对手有 1 项活跃临床试验，建议持续跟踪
  - 识别到 6 个 empty white-space 组合，建议评估进入策略
  - 识别到 9 个 minimal 覆盖组合，建议持续观察
  - 补充数据缺口，完善竞争对手档案


In [3]:
# Cell 3: Technology Brief + 三个质量能力区块
strategies = get_all_strategies(driver)
target = strategies[0]
print(f"使用策略: {target['strategy_type']}\n")

t_brief = generate_technology_brief(
    driver, strategy_id=target["id"],
    technology_question=f"{target['strategy_type']} 的公开证据成熟度如何？",
    persist=True,
)
t_bid = t_brief["brief_id"]
print(f"✅ Technology Brief: {t_bid}\n")

with driver.session() as s:
    row = s.run("""
        MATCH (b:IntelligenceProduct {brief_id: $bid})
        RETURN b.source_credibility AS src,
               b.conflict_warnings AS cf,
               b.related_patent_mappings AS pm
    """, bid=t_bid).single()

print("=== source_credibility ===")
print(json.dumps(json.loads(row["src"] or "{}"), ensure_ascii=False, indent=2))

print("\n=== conflict_warnings ===")
cf = json.loads(row["cf"] or "{}")
print(f"  total_conflicts: {cf.get('total_conflicts', 0)}")
for c in cf.get("claim_status_conflicts", [])[:3]:
    print(f"    [claim_status] {c.get('construct_name')}: {c.get('claim_type')}")

print("\n=== related_patent_mappings ===")
pm = json.loads(row["pm"] or "{}")
print(f"  total_mappings: {pm.get('total_mappings', 0)}")
for m in pm.get("mappings", [])[:3]:
    print(f"    {m.get('patent_title')} -> {m.get('claim_text', '')[:60]}")

print("\n=== recommended_next_steps ===")
for s_ in t_brief.get("recommended_next_steps", []):
    print(f"  - {s_}")

使用策略: biofilm_disruption

✅ Technology Brief: CI:BRIEF:0640CC20

=== source_credibility ===
{
  "total_sources": 1,
  "by_tier": {
    "primary": 0,
    "secondary": 1,
    "tertiary": 0,
    "unknown": 0
  },
  "note": "来源可信度只反映渠道可靠性，不代表内容正确性。"
}

=== conflict_warnings ===
  total_conflicts: 0

=== related_patent_mappings ===
  total_mappings: 0

=== recommended_next_steps ===
  - 存在技术主张但缺少实验/研究结果，建议补充体外或体内数据验证。
  - 当前证据成熟度均停留在体外阶段，临床转化存在明显不确定性。
  - 本简报涉及 1 个策略，已附同类策略横向对比矩阵
  - 范围内识别到 2 个 empty white-space 组合，建议评估进入策略


In [4]:
# Cell 4: 全量验证
with driver.session() as s:
    print("=== Competitor Brief 质量能力字段覆盖 ===")
    for r in s.run("""
        MATCH (b:IntelligenceProduct {brief_type: 'competitor'})
        RETURN b.brief_id AS bid,
               b.source_credibility_breakdown AS sc,
               b.entity_resolution_hints AS er,
               b.conflict_summary AS cf
    """):
        has = lambda x: bool(x and x not in ("", "{}"))
        print(f"  {r['bid']}  sc={has(r['sc'])}  er={has(r['er'])}  cf={has(r['cf'])}")

    print("\n=== Technology Brief 质量能力字段覆盖 ===")
    for r in s.run("""
        MATCH (b:IntelligenceProduct {brief_type: 'engineered_phage_technology'})
        RETURN b.brief_id AS bid,
               b.source_credibility AS src,
               b.conflict_warnings AS cf,
               b.related_patent_mappings AS pm
    """):
        has = lambda x: bool(x and x not in ("", "{}"))
        print(f"  {r['bid']}  src={has(r['src'])}  cf={has(r['cf'])}  pm={has(r['pm'])}")

=== Competitor Brief 质量能力字段覆盖 ===
  CI:BRIEF:EB82EFAA  sc=False  er=False  cf=False
  CI:BRIEF:1CD36BA7  sc=False  er=False  cf=False
  CI:BRIEF:C19F7DC9  sc=False  er=False  cf=False
  CI:BRIEF:A2AFD357  sc=False  er=False  cf=False
  CI:BRIEF:890A2449  sc=False  er=False  cf=False
  CI:BRIEF:171958BA  sc=False  er=False  cf=False
  CI:BRIEF:B7B48C7D  sc=False  er=False  cf=False
  CI:BRIEF:06177347  sc=False  er=False  cf=False
  CI:BRIEF:1639FBEC  sc=False  er=False  cf=False
  CI:BRIEF:5DB308F0  sc=True  er=True  cf=True
  CI:BRIEF:81D6988E  sc=True  er=True  cf=True
  CI:BRIEF:DB3A21A0  sc=True  er=True  cf=True

=== Technology Brief 质量能力字段覆盖 ===
  CI:BRIEF:928BD96E  src=False  cf=False  pm=False
  CI:BRIEF:DB0CB2A3  src=False  cf=False  pm=False
  CI:BRIEF:EF1C3D46  src=False  cf=False  pm=False
  CI:BRIEF:6DE5BDE4  src=False  cf=False  pm=False
  CI:BRIEF:7A442CBD  src=False  cf=False  pm=False
  CI:BRIEF:8AD032AC  src=False  cf=False  pm=False
  CI:BRIEF:FA60BDE1  src=False  cf